In [ ]:
import pandas as pd
import numpy as np
import re

# =========================
# LOAD DATA
# =========================
file_path = "srcsc-2026-claims-equipment-failure (2).xlsx"
df_freq = pd.read_excel(file_path, sheet_name="freq")
df_sev  = pd.read_excel(file_path, sheet_name="sev")

# simpan snapshot BEFORE (biar bisa bandingkan repaired, missing, dll)
df_freq_before = df_freq.copy()
df_sev_before  = df_sev.copy()

# =========================
# RULES
# =========================
NUMERIC_RANGES = {
    "maintenance_int": (100, 5000),
    "usage_int": (0, 24),
    "exposure": (0, 1),
    "claim_count": (0, 3),
    "claim_amount": (11000, 790000),
}

EQUIPMENT_TYPE_ALLOWED = {
    "Quantum Bore",
    "Graviton Extractor",
    "Ion Pulverizer",
    "ReglAggregators",
    "FexStram Carrier",
    "Flux Rider",
}

SOLAR_SYSTEM_ALLOWED = {"Helionis Cluster", "Epsilon", "Zeta"}

# suffix pasti: _??? + 4 digit, hanya di belakang (end-of-string)
SUFFIX_PATTERN = re.compile(r"_\?{3}\d{4}$")

def strip_suffix_qqq4(x):
    """Hapus suffix _???dddd kalau ada di akhir string."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = SUFFIX_PATTERN.sub("", s).strip()
    return s if s != "" else np.nan

def clean_equipment_id(x):
    s = strip_suffix_qqq4(x)
    if pd.isna(s): return np.nan
    return s if re.fullmatch(r"EQ-\d{6}", s) else np.nan

def clean_policy_id(x):
    s = strip_suffix_qqq4(x)
    if pd.isna(s): return np.nan
    return s if re.fullmatch(r"EF-\d{6}", s) else np.nan

def clean_equipment_type(x):
    s = strip_suffix_qqq4(x)
    if pd.isna(s):
        return np.nan
    s = s.strip()
    return s if s in EQUIPMENT_TYPE_ALLOWED else np.nan

def clean_solar_system(x):
    s = strip_suffix_qqq4(x)
    if pd.isna(s): return np.nan
    return s if s in SOLAR_SYSTEM_ALLOWED else np.nan


def apply_cleaning(df):
    df2 = df.copy()

    # =========================
    # CLEAN equipment_age (RULE BARU)
    # =========================
    if "equipment_age" in df2.columns:
        df2["equipment_age"] = pd.to_numeric(df2["equipment_age"], errors="coerce")
        df2.loc[df2["equipment_age"] < 0, "equipment_age"] = np.nan

    # =========================
    # numeric cleaning lain
    # =========================
    for col, (lo, hi) in NUMERIC_RANGES.items():
        if col not in df2.columns:
            continue

        df2[col] = pd.to_numeric(df2[col], errors="coerce")

        # ---- PERLAKUAN KHUSUS claim_amount ----
        if col == "claim_amount":
            # kalau < batas bawah -> NaN
            df2.loc[df2[col] < lo, col] = np.nan
            # kalau > batas atas -> cap ke batas atas (bukan NaN)
            df2.loc[df2[col] > hi, col] = hi
        else:
            # default: out of range -> NaN
            df2.loc[(df2[col] < lo) | (df2[col] > hi), col] = np.nan

    # =========================
    # categorical cleaning
    # =========================
    if "equipment_id" in df2.columns:
        df2["equipment_id"] = df2["equipment_id"].apply(clean_equipment_id)

    if "policy_id" in df2.columns:
        df2["policy_id"] = df2["policy_id"].apply(clean_policy_id)

    if "equipment_type" in df2.columns:
        df2["equipment_type"] = (
            df2["equipment_type"]
            .astype("string")
            .str.replace(r"_\?{3}\d{4}$", "", regex=True)
            .str.strip()
        )
        df2["equipment_type"] = df2["equipment_type"].apply(clean_equipment_type)

    if "solar_system" in df2.columns:
        df2["solar_system"] = df2["solar_system"].apply(clean_solar_system)

    return df2


df_freq_clean = apply_cleaning(df_freq)
df_sev_clean  = apply_cleaning(df_sev)


# =========================
# MISSING BEFORE/AFTER PER VARIABLE
# =========================
def missing_per_variable(df_before, df_after, sheet_name):
    out = pd.DataFrame({
        "variable": df_before.columns,
        "missing_before": df_before.isna().sum().values,
        "missing_after": df_after.isna().sum().values,
    })
    out["additional_missing"] = out["missing_after"] - out["missing_before"]
    out["sheet"] = sheet_name
    return out.sort_values(["additional_missing","missing_after"], ascending=[False, False]).reset_index(drop=True)

missing_freq = missing_per_variable(df_freq_before, df_freq_clean, "freq")
missing_sev  = missing_per_variable(df_sev_before, df_sev_clean, "sev")

print("=== Missing per variable (freq) ===")
print(missing_freq)

print("\n=== Missing per variable (sev) ===")
print(missing_sev)


# =========================
# TOTAL MISSING SUMMARY
# =========================
total_summary = pd.DataFrame({
    "sheet": ["freq", "sev"],
    "total_missing_before": [
        int(df_freq_before.isna().sum().sum()),
        int(df_sev_before.isna().sum().sum())
    ],
    "total_missing_after": [
        int(df_freq_clean.isna().sum().sum()),
        int(df_sev_clean.isna().sum().sum())
    ]
})

total_summary["additional_missing"] = (
    total_summary["total_missing_after"]
    - total_summary["total_missing_before"]
)

print("\n=== Total missing summary ===")
print(total_summary)


# =========================
# DIAGNOSTIC: repaired count
# =========================
def repaired_count(series_before, series_after):
    b = series_before.astype("string")
    a = series_after.astype("string")

    had_suffix = b.fillna("").str.contains(r"_\?{3}\d{4}$", regex=True)
    repaired = had_suffix & a.notna()

    return int(had_suffix.sum()), int(repaired.sum())


for col in ["equipment_id", "policy_id", "equipment_type", "solar_system"]:
    if col in df_freq_before.columns:
        had, rep = repaired_count(df_freq_before[col], df_freq_clean[col])
        print(f"\n[freq] {col}: had_suffix={had}, repaired={rep}")

    if col in df_sev_before.columns:
        had, rep = repaired_count(df_sev_before[col], df_sev_clean[col])
        print(f"[sev ] {col}: had_suffix={had}, repaired={rep}")


# =========================
# EXPORT CLEAN DATA (FIXED)
# =========================
output_file = "cleaned_equipment_failure_data.xlsx"

try:
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df_freq_clean.to_excel(writer, sheet_name="freq_clean", index=False)
        df_sev_clean.to_excel(writer, sheet_name="sev_clean", index=False)

    print("\nFile berhasil disimpan:", output_file)

except ModuleNotFoundError as e:
    print("\nGagal export ke Excel karena engine tidak tersedia:", e)
    print("Fallback: export ke CSV...")

    df_freq_clean.to_csv("freq_clean.csv", index=False)
    df_sev_clean.to_csv("sev_clean.csv", index=False)

    print("File berhasil disimpan: freq_clean.csv dan sev_clean.csv")

=== Missing per variable (freq) ===
          variable  missing_before  missing_after  additional_missing sheet
0         exposure             125            415                 290  freq
1        usage_int             141            409                 268  freq
2  maintenance_int             154            413                 259  freq
3    equipment_age             144            288                 144  freq
4      claim_count             175            205                  30  freq
5   equipment_type             239            239                   0  freq
6     solar_system             234            234                   0  freq
7        policy_id             224            224                   0  freq
8     equipment_id             221            221                   0  freq

=== Missing per variable (sev) ===
           variable  missing_before  missing_after  additional_missing sheet
0          exposure              13             40                  27   sev
1         usag

In [ ]:
output_file = "cleaned_equipment_failure_data.xlsx"

with pd.ExcelWriter(output_file) as writer:
    df_freq_clean.to_excel(writer, sheet_name="freq_clean", index=False)
    df_sev_clean.to_excel(writer, sheet_name="sev_clean", index=False)

print("File berhasil dibuat:", output_file)

File berhasil dibuat: cleaned_equipment_failure_data.xlsx


In [ ]:
from google.colab import files

files.download("cleaned_equipment_failure_data.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>